# Notebook 00 — Environment Check

**Purpose:** verify the repository root, Conda environment, required scientific packages, CUDA/GPU visibility, OpenCV, WSL/Linux, Git state, and environment evidence for checkpoint 00. This notebook performs diagnostics only.

## CONFIG and project root detection

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / 'FISH_AI_PROJECT_WORKFLOW.md').is_file(), 'Project root marker missing'
assert (PROJECT_ROOT / 'AGENTS.md').is_file(), 'AGENTS.md missing'
LOG_DIR = PROJECT_ROOT / 'logs' / 'environment'
ENVIRONMENT_DIR = PROJECT_ROOT / 'environment'
LOG_DIR.mkdir(parents=True, exist_ok=True)
ENVIRONMENT_DIR.mkdir(parents=True, exist_ok=True)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')

PROJECT_ROOT: /home/diy-hus/fish


## Python / Conda information

In [2]:
import os
import shutil
import sys

python_info = {
    'Python executable': sys.executable,
    'Python version': sys.version.replace('\n', ' '),
    'sys.prefix': sys.prefix,
    'Conda environment': os.environ.get('CONDA_DEFAULT_ENV', 'not set'),
    'CONDA_PREFIX': os.environ.get('CONDA_PREFIX', 'not set'),
    'pip path': shutil.which('pip') or 'not found',
}
for key, value in python_info.items():
    print(f'{key}: {value}')
assert python_info['Conda environment'] == 'fish', 'Notebook is not running in the fish environment'
assert Path(sys.executable).parent.name == 'bin' and Path(sys.executable).parent.parent.name == 'fish'

Python executable: /home/diy-hus/miniconda3/envs/fish/bin/python
Python version: 3.11.15 (main, Jun 11 2026, 15:20:16) [GCC 14.3.0]
sys.prefix: /home/diy-hus/miniconda3/envs/fish
Conda environment: fish
CONDA_PREFIX: /home/diy-hus/miniconda3/envs/fish
pip path: /home/diy-hus/miniconda3/envs/fish/bin/pip


## Package versions

In [3]:
import cv2
import ipykernel
import matplotlib
import nbconvert
import numpy as np
import pandas as pd
import torch
import torchvision
import ultralytics
import yaml

package_versions = {
    'ipykernel': ipykernel.__version__,
    'nbconvert': nbconvert.__version__,
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'ultralytics': ultralytics.__version__,
    'OpenCV': cv2.__version__,
    'NumPy': np.__version__,
    'pandas': pd.__version__,
    'matplotlib': matplotlib.__version__,
    'PyYAML': yaml.__version__,
}
for name, version in package_versions.items():
    print(f'{name}: {version}')

ipykernel: 7.3.0
nbconvert: 7.17.1
torch: 2.13.0+cu130
torchvision: 0.28.0+cu130
ultralytics: 8.4.120
OpenCV: 5.0.0
NumPy: 2.4.6
pandas: 3.0.5
matplotlib: 3.11.1
PyYAML: 6.0.3


## PyTorch / CUDA / GPU

In [4]:
cuda_available = torch.cuda.is_available()
torch_cuda_version = torch.version.cuda
gpu_name = torch.cuda.get_device_name(0) if cuda_available else 'Not available'
print(f'PyTorch: {torch.__version__}')
print(f'CUDA runtime reported by PyTorch: {torch_cuda_version}')
print(f'CUDA available: {cuda_available}')
print(f'GPU name: {gpu_name}')

PyTorch: 2.13.0+cu130
CUDA runtime reported by PyTorch: 13.0
CUDA available: True
GPU name: NVIDIA GeForce RTX 3050


## OpenCV

In [5]:
test_image = np.zeros((8, 8, 3), dtype=np.uint8)
gray_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2GRAY)
assert gray_image.shape == (8, 8)
print(f'OpenCV {cv2.__version__}: basic array conversion passed')

OpenCV 5.0.0: basic array conversion passed


## System / WSL information

In [6]:
import platform
import socket

system_info = {
    'hostname': socket.gethostname(),
    'system': platform.system(),
    'release': platform.release(),
    'platform': platform.platform(),
    'WSL detected': 'microsoft' in platform.release().lower(),
}
for key, value in system_info.items():
    print(f'{key}: {value}')

hostname: diy-hus
system: Linux
release: 6.18.33.2-microsoft-standard-WSL2
platform: Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
WSL detected: True


## Git information

In [7]:
import subprocess

def git_output(*args):
    result = subprocess.run(
        ['git', *args], cwd=PROJECT_ROOT, text=True, capture_output=True, check=False
    )
    return (result.stdout or result.stderr).strip()

git_branch = git_output('symbolic-ref', '--short', 'HEAD')
git_status = git_output('status', '--short', '--branch')
print(f'Branch: {git_branch}')
print(git_status)

Branch: master
## No commits yet on master
?? .gitignore
?? .vscode/
?? AGENTS.md
?? FISH_AI_PROJECT_WORKFLOW.md
?? configs/
?? environment/
?? logs/
?? notebooks/


## Environment export

In [8]:
environment_yml = ENVIRONMENT_DIR / 'environment.yml'
pip_freeze = ENVIRONMENT_DIR / 'pip_freeze.txt'
assert environment_yml.is_file(), 'environment.yml has not been exported'
assert pip_freeze.is_file(), 'pip_freeze.txt has not been exported'
print(f'Environment history: {environment_yml.relative_to(PROJECT_ROOT)}')
print(f'Pip freeze: {pip_freeze.relative_to(PROJECT_ROOT)}')

Environment history: environment/environment.yml
Pip freeze: environment/pip_freeze.txt


## Final summary / checkpoint result

In [9]:
from datetime import datetime, timezone

summary = {
    'datetime': datetime.now(timezone.utc).isoformat(),
    'hostname': system_info['hostname'],
    'WSL/Linux information': system_info['platform'],
    'Python executable': python_info['Python executable'],
    'Python version': python_info['Python version'],
    'Conda environment': python_info['Conda environment'],
    'PyTorch version': package_versions['torch'],
    'CUDA version': str(torch_cuda_version),
    'CUDA available': str(cuda_available),
    'GPU name': gpu_name,
    'Ultralytics version': package_versions['ultralytics'],
    'OpenCV version': package_versions['OpenCV'],
    'NumPy version': package_versions['NumPy'],
    'pandas version': package_versions['pandas'],
    'matplotlib version': package_versions['matplotlib'],
    'Git branch': git_branch,
    'Checkpoint result': 'PASS' if cuda_available else 'PASS with CUDA unavailable warning',
}
summary_path = LOG_DIR / 'environment_summary.txt'
summary_path.write_text(
    ''.join(f'{key}: {value}\n' for key, value in summary.items()), encoding='utf-8'
)
for key, value in summary.items():
    print(f'{key}: {value}')
print(f'Summary saved: {summary_path.relative_to(PROJECT_ROOT)}')

datetime: 2026-08-17T03:30:21.342245+00:00
hostname: diy-hus
WSL/Linux information: Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
Python executable: /home/diy-hus/miniconda3/envs/fish/bin/python
Python version: 3.11.15 (main, Jun 11 2026, 15:20:16) [GCC 14.3.0]
Conda environment: fish
PyTorch version: 2.13.0+cu130
CUDA version: 13.0
CUDA available: True
GPU name: NVIDIA GeForce RTX 3050
Ultralytics version: 8.4.120
OpenCV version: 5.0.0
NumPy version: 2.4.6
pandas version: 3.0.5
matplotlib version: 3.11.1
Git branch: master
Checkpoint result: PASS
Summary saved: logs/environment/environment_summary.txt
